# Development and Testing of the Retreiver

In [4]:
# Importing packages
import pandas as pd
import torch
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, logging, AutoModel
logging.set_verbosity_error()
import random
import re
import numpy as np
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
from typing import List
from dotenv import load_dotenv
import os
from torch import Tensor
import faiss 
import json

In [ ]:
# Loading token
load_dotenv('token.env')
token = os.getenv('HUGGINGFACE_TOKEN')

# Loading model - pass token directly
model_name = "meta-llama/Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name, token=token)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map="auto",
    token=token  # Pass token here
)

In [12]:
df = pd.read_csv("/work/mbouthil/projects/research_project/MEDRAG/synthetic_data/synq.csv")
queries = df['QUERY'].tolist()
passages = df['PASSAGE'].tolist()

### Loading Query Encoder

In [5]:
# Loading Query Encoder
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
query_encoder = AutoModel.from_pretrained("bert-base-uncased")

query_encoder = AutoModel.from_pretrained(
    "/work/mbouthil/projects/research_project/MEDRAG/model_weights/query_encoder"
) # .to("cuda")

query_encoder.eval()

def encode_query(query:str, batch_size:int=32) -> Tensor:

    embeddings = []

    with torch.no_grad():
        inputs = tokenizer(
            query, 
            padding=True,
            truncation=True,
            return_tensors="pt",
            max_length=512
        ) #.to("cuda")

    outputs = query_encoder(**inputs)
    cls_embeddings = outputs.last_hidden_state[:, 0] 

    embeddings.append(cls_embeddings.cpu())

    return torch.cat(embeddings, 0)

### Loading Passage Data

In [17]:
# Loading Index
index = faiss.read_index("/work/mbouthil/projects/research_project/MEDRAG/retrieval_data/passage.index")

# Load metadata
metadata = []
with open("/work/mbouthil/projects/research_project/MEDRAG/retrieval_data/passage_metadata.jsonl") as f:
    for line in f:
        metadata.append(json.loads(line))

# Testing

In [13]:
print(queries[0])
print("/n/n")
print(passages[0])

Does the patient have a history of tuberculosis prior to this admission?
/n/n
Admission Date:  [**2151-7-16**]       Discharge Date:  [**2151-8-4**]


Service:
ADDENDUM:

RADIOLOGIC STUDIES:  Radiologic studies also included a chest
CT, which confirmed cavitary lesions in the left lung apex
consistent with infectious process/tuberculosis. This also
moderate-sized left pleural effusion.


In [14]:
question = 'Subject ID: ' + str(df['SUBJECT_ID'].iloc[0]) + '\n'  +  "Does the patient have a history of tuberculosis?"

In [15]:
query_emb = encode_query([question]).detach().cpu().numpy()

In [19]:
K = 5
scores, ids = index.search(query_emb, K)

In [23]:
candidates = [metadata[i]["text"] for i in ids[0]]
for i in candidates:
    print(i)
    print("\n\n\n")

Subject ID: 14879
Type 2 diabetes, uncontrolled. 5. Anemia, acute blood loss. 6. Lymphoma. 7. Failure to thrive and deconditioning. DISCHARGE MEDICATIONS:
1. Tylenol 325-650 mg po q 4-6 h prn pain. 2. Pantoprazole 40 mg po qd. 3. Heparin subcu 5,000 U q 8 h. 4. Citalopram 20 mg po qd. 5. Mirtazapine 50 mg po q hs. 6. Epoetin Alfa 4,000 U 2 x week--Monday, Thursday. 7. Colace 100 mg po bid--hold for loose stools. 8.




Subject ID: 11018
- Continue long steroid taper at home (Prednisone 60mg X 7 days,
40mg X 7 days, 20 mg X7 days, 10mg X 7 days, off)
- Continue supplemental oxygen, albuterol and ipratropium nebs
- Continue MS contin and morphine liquid PRN for air hunger,
shortness of breath
- Continue lorazepam PRN for air hunger, shortness of breath,
anxiety
. # HIV: Down trending CD4 count, ?due to acute illness.




Subject ID: 68109
However, there is circumferential wall
thickening involving both ureters throughout its course. These
findings are overall suggestive of bilateral pyel

# Touble Shooting

In [10]:
bad_df = pd.read_csv("/work/mbouthil/projects/research_project/MEDRAG/synthetic_data/bad_data.csv")
asynq_df = pd.read_csv("/work/mbouthil/projects/research_project/MEDRAG/synthetic_data/add_synq.csv")

In [24]:
asynq_df.head(20)

,ROW_ID,SUBJECT_ID,HADM_ID,CHARTDATE,CHARTTIME,STORETIME,CATEGORY,DESCRIPTION,CGID,ISERROR,TEXT,NOTE,QUERY,PASSAGE
0,174,22532,167853,2151-08-04,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2151-7-16**] Dischar...,Admission Date: [**2151-7-16**] Dischar...,Does the patient have a history of tuberculosi...,Admission Date: [**2151-7-16**] Dischar...
1,174,22532,167853,2151-08-04,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2151-7-16**] Dischar...,Admission Date: [**2151-7-16**] Dischar...,Does the patient have a past history of TB?,Admission Date: [**2151-7-16**] Dischar...
2,174,22532,167853,2151-08-04,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2151-7-16**] Dischar...,Admission Date: [**2151-7-16**] Dischar...,Is there a history of tuberculosis in the pati...,Admission Date: [**2151-7-16**] Dischar...
3,174,22532,167853,2151-08-04,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2151-7-16**] Dischar...,Admission Date: [**2151-7-16**] Dischar...,Has the patient ever been diagnosed with tuber...,Admission Date: [**2151-7-16**] Dischar...
4,174,22532,167853,2151-08-04,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2151-7-16**] Dischar...,Admission Date: [**2151-7-16**] Dischar...,Is tuberculosis a part of the patient's medica...,Admission Date: [**2151-7-16**] Dischar...
5,174,22532,167853,2151-08-04,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2151-7-16**] Dischar...,Admission Date: [**2151-7-16**] Dischar...,Does the patient have a pre-existing diagnosis...,Admission Date: [**2151-7-16**] Dischar...
6,174,22532,167853,2151-08-04,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2151-7-16**] Dischar...,Admission Date: [**2151-7-16**] Dischar...,Is there any prior history of tuberculosis in ...,Admission Date: [**2151-7-16**] Dischar...
7,174,22532,167853,2151-08-04,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2151-7-16**] Dischar...,Admission Date: [**2151-7-16**] Dischar...,Has the patient been previously diagnosed with...,Admission Date: [**2151-7-16**] Dischar...
8,174,22532,167853,2151-08-04,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2151-7-16**] Dischar...,HEAD CT: Head CT showed no intracranial hemor...,Has the patient had any previous imaging studi...,HEAD CT: Head CT showed no intracranial hemor...
9,174,22532,167853,2151-08-04,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2151-7-16**] Dischar...,HEAD CT: Head CT showed no intracranial hemor...,Has the patient undergone any prior imaging or...,HEAD CT: Head CT showed no intracranial hemor...


In [11]:
print(len(bad_df) == len(asynq_df))


True
